# Recurrent Neural Networks



## RNN - 'ShakesGen'

Let's create a `ShakesGen` !!<br><br>
The data folder contains a shakespeare folder with works from William Shakespeare. The task is to implement an RNN that learns to write Shakespeare-style text.

* The Corpus class serves as a dataset, and you can retrieve a batch with its target by calling `get_batch` on a batchified dataset.

* Build the missing model components and train your ShakesGen model.
* Generate at least 30 lines of text using your ShakesGen model.

Especially, if you train on cpu, you can stop training after 5 minutes and generate based on the current model state.

In [ ]:
from IPython.display import Image
from IPython.core.display import HTML
Image(url= "https://miro.medium.com/max/4000/0*WdbXF_e8kZI1R5nQ.png", width=700)

In [ ]:
# Some imports
import torch
import torch.nn as nn
import torch.autograd as autograd
import torch.cuda as cuda
import torch.optim as optim
import torch.nn.functional as F
import os
import tqdm
import numpy as np

In [ ]:
class Dictionary(object):
    def __init__(self):
        self.word2idx = {}
        self.idx2word = []

    def add_word(self, word):
        if word not in self.word2idx:
            self.idx2word.append(word)
            self.word2idx[word] = len(self.idx2word) - 1
        return self.word2idx[word]

    def __len__(self):
        return len(self.idx2word)


class Corpus(object):
    def __init__(self, path):
        self.dictionary = Dictionary()

        # This is very english language specific
        # We will ingest only these characters:
        self.whitelist = [chr(i) for i in range(32, 127)]

        self.train = self.tokenize(os.path.join(path, 'train.txt'))
        self.valid = self.tokenize(os.path.join(path, 'valid.txt'))

    def tokenize(self, path):
        """Tokenizes a text file."""
        assert os.path.exists(path)
        # Add words to the dictionary
        with open(path, 'r',  encoding="utf8") as f:
            tokens = 0
            for line in f:
                line = ''.join([c for c in line if c in self.whitelist])
                words = line.split() + ['<eos>']
                tokens += len(words)
                for word in words:
                    self.dictionary.add_word(word)

        # Tokenize file content
        with open(path, 'r',  encoding="utf8") as f:
            ids = torch.LongTensor(tokens)
            token = 0
            for line in f:
                line = ''.join([c for c in line if c in self.whitelist])
                words = line.split() + ['<eos>']
                for word in words:
                    ids[token] = self.dictionary.word2idx[word]
                    token += 1

        return ids

def batchify(data, batch_size):
    # Work out how cleanly we can divide the dataset into bsz parts.
    nbatch = data.size(0) // batch_size
    # Trim off any extra elements that wouldn't cleanly fit (remainders).
    data = data.narrow(0, 0, nbatch * batch_size)
    # Evenly divide the data across the bsz batches.
    data = data.view(batch_size, -1).t().contiguous()
    return data

def get_batch(source, i, bptt_size=35):
    seq_len = min(bptt_size, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].view(-1)
    return data, target

In [ ]:
# Run this cell first to download and prepare the dataset

import os
import requests

# Create directory structure
os.makedirs('./data/shakespeare', exist_ok=True)

# Download the Tiny Shakespeare dataset
print("Downloading Shakespeare dataset...")
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)

if response.status_code == 200:
    full_text = response.text
    print(f"Downloaded {len(full_text):,} characters")

    # Split into train (90%) and validation (10%) sets
    split_idx = int(len(full_text) * 0.9)
    train_text = full_text[:split_idx]
    valid_text = full_text[split_idx:]

    # Save train.txt
    with open('./data/shakespeare/train.txt', 'w', encoding='utf-8') as f:
        f.write(train_text)
    print(f"Created train.txt with {len(train_text):,} characters")

    # Save valid.txt
    with open('./data/shakespeare/valid.txt', 'w', encoding='utf-8') as f:
        f.write(valid_text)
    print(f"Created valid.txt with {len(valid_text):,} characters")

    print("\n✓ Dataset ready! You can now run: corpus = Corpus('./data/shakespeare')")
else:
    print(f"Error downloading dataset: {response.status_code}")


Downloaded 1,115,394 characters
Created train.txt with 1,003,854 characters
Created valid.txt with 111,540 characters

✓ Dataset ready! You can now run: corpus = Corpus('./data/shakespeare')


In [ ]:
# Use Corpus to load data
corpus = Corpus('./data/shakespeare')

In [ ]:
vocab_size = len(corpus.dictionary)
print(vocab_size)

# Print first 100 words from training data
words = [corpus.dictionary.idx2word[corpus.train[i].item()] for i in range(min(100, len(corpus.train)))]
print(' '.join(words))

25672
First Citizen: <eos> Before we proceed any further, hear me speak. <eos> <eos> All: <eos> Speak, speak. <eos> <eos> First Citizen: <eos> You are all resolved rather to die than to famish? <eos> <eos> All: <eos> Resolved. resolved. <eos> <eos> First Citizen: <eos> First, you know Caius Marcius is chief enemy to the people. <eos> <eos> All: <eos> We know't, we know't. <eos> <eos> First Citizen: <eos> Let us kill him, and we'll have corn at our own price. <eos> Is't a verdict? <eos> <eos> All: <eos> No more talking on't; let it be done: away, away! <eos> <eos> Second


In [ ]:
idx = corpus.dictionary.word2idx.get("That", -1)  # returns -1 if not found
print(f"Index of the word 'That': {idx}")

Index of the word 'That': 409


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# TODO: Implement RNN model class & Training loop here

# Model Definition
class ShakesGenRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super(ShakesGenRNN, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

        # Embedding layer to convert word indices to dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # RNN layer (using LSTM for better long-term dependencies)
        self.rnn = nn.LSTM(embed_dim, hidden_dim, num_layers,
                          dropout=dropout, batch_first=False)

        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)

        # Output layer to predict next word
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden):
        # x shape: (seq_len, batch_size)
        embedded = self.dropout(self.embedding(x))
        output, hidden = self.rnn(embedded, hidden)
        output = self.dropout(output)
        decoded = self.fc(output)
        return decoded, hidden

    def init_hidden(self, batch_size):
        weight = next(self.parameters())
        return (weight.new_zeros(self.num_layers, batch_size, self.hidden_dim),
                weight.new_zeros(self.num_layers, batch_size, self.hidden_dim))


# Hyperparameters
batch_size = 32
bptt_size = 35  # Backpropagation through time sequence length
embed_dim = 128
hidden_dim = 256
num_layers = 2
dropout = 0.3
learning_rate = 0.001
epochs = 5

# Prepare data
train_data = batchify(corpus.train, batch_size).to(device)
val_data = batchify(corpus.valid, batch_size).to(device)

# Initialize model
model = ShakesGenRNN(vocab_size, embed_dim, hidden_dim, num_layers, dropout).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training on: {device}")


# Training function
def train_epoch(model, data, criterion, optimizer, bptt_size):
    model.train()
    total_loss = 0
    hidden = model.init_hidden(batch_size)

    for i in range(0, data.size(0) - 1, bptt_size):
        # Get batch
        inputs, targets = get_batch(data, i, bptt_size)

        # Detach hidden state from computational graph
        hidden = tuple(h.detach() for h in hidden)

        # Forward pass
        optimizer.zero_grad()
        output, hidden = model(inputs, hidden)

        # Calculate loss
        loss = criterion(output.view(-1, vocab_size), targets)

        # Backward pass
        loss.backward()

        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)

        optimizer.step()

        total_loss += loss.item()

    return total_loss / (data.size(0) // bptt_size)


# Validation function
def evaluate(model, data, criterion, bptt_size):
    model.eval()
    total_loss = 0
    hidden = model.init_hidden(batch_size)

    with torch.no_grad():
        for i in range(0, data.size(0) - 1, bptt_size):
            inputs, targets = get_batch(data, i, bptt_size)
            output, hidden = model(inputs, hidden)
            loss = criterion(output.view(-1, vocab_size), targets)
            total_loss += loss.item()
            hidden = tuple(h.detach() for h in hidden)

    return total_loss / (data.size(0) // bptt_size)


# Training loop
print("\nStarting training...")
best_val_loss = float('inf')

for epoch in range(epochs):
    train_loss = train_epoch(model, train_data, criterion, optimizer, bptt_size)
    val_loss = evaluate(model, val_data, criterion, bptt_size)

    print(f'Epoch {epoch+1:2d} | Train Loss: {train_loss:.3f} | '
          f'Train PPL: {np.exp(train_loss):7.3f} | Val Loss: {val_loss:.3f} | '
          f'Val PPL: {np.exp(val_loss):7.3f}')

    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_shakesgen_model.pt')

print("\nTraining completed!")


Model parameters: 10,805,320
Training on: cuda

Starting training...
Epoch  1 | Train Loss: 7.224 | Train PPL: 1372.404 | Val Loss: 7.287 | Val PPL: 1460.717
Epoch  2 | Train Loss: 6.567 | Train PPL: 710.892 | Val Loss: 6.930 | Val PPL: 1022.852
Epoch  3 | Train Loss: 6.228 | Train PPL: 506.645 | Val Loss: 6.889 | Val PPL: 981.716
Epoch  4 | Train Loss: 6.052 | Train PPL: 424.806 | Val Loss: 6.848 | Val PPL: 942.153
Epoch  5 | Train Loss: 5.899 | Train PPL: 364.834 | Val Loss: 6.791 | Val PPL: 890.030

Training completed!


In [ ]:
# Load best model (from epoch with lowest validation loss) before generating
model.load_state_dict(torch.load('best_shakesgen_model.pt'))


# Text generation function
def generate_text(model, corpus, start_text="The", num_words=300, temperature=0.6):
    """
    Generate text using the trained model

    Args:
        model: Trained RNN model
        corpus: Corpus object with dictionary
        start_text: Starting word(s)
        num_words: Number of words to generate
        temperature: Controls randomness (lower = more conservative)
    """
    model.eval()

    # Tokenize start text
    words = start_text.split()
    input_seq = torch.LongTensor([[corpus.dictionary.word2idx.get(w, 0) for w in words]]).to(device)
    input_seq = input_seq.t()  # Transpose to (seq_len, batch_size)

    hidden = model.init_hidden(1)
    generated_words = words.copy()

    with torch.no_grad():
        # Process start text
        for i in range(len(words) - 1):
            output, hidden = model(input_seq[i:i+1], hidden)

        # Generate new words
        input_word = input_seq[-1:] if len(words) > 0 else torch.LongTensor([[0]]).to(device)

        for _ in range(num_words):
            output, hidden = model(input_word, hidden)

            # Apply temperature scaling
            word_weights = output.squeeze().div(temperature).exp()
            word_idx = torch.multinomial(word_weights, 1)[0].item()

            # Add generated word
            generated_words.append(corpus.dictionary.idx2word[word_idx])

            # Update input
            input_word = torch.LongTensor([[word_idx]]).to(device)

    # Format output with line breaks
    text = ' '.join(generated_words)
    text = text.replace(' , ', ', ').replace(' .', '.').replace(' !', '!').replace(' ?', '?')

    return text


# Generate Shakespeare-style text (at least 30 lines)
print("\n" + "="*80)
print("GENERATED SHAKESPEARE TEXT")
print("="*80 + "\n")

generated_text = generate_text(model, corpus, start_text="The", num_words=300, temperature=0.8)
print(generated_text)

# Try different starting prompts
print("\n" + "="*80 + "\n")
generated_text2 = generate_text(model, corpus, start_text="To be", num_words=200, temperature=0.7)
print(generated_text2)

print("\n" + "="*80 + "\n")
generated_text3 = generate_text(model, corpus, start_text="Lord", num_words=250, temperature=0.9)
print(generated_text3)



GENERATED SHAKESPEARE TEXT

The velvet. <eos> <eos> QUEEN Citizen: <eos> Yes, thou O noble to have I my BALTHASAR: <eos> And shall shall a own son of the brother <eos> All he that from have the thousand free <eos> <eos> ISABELLA: <eos> I have the thousand pardon to a noble have: <eos> And thou 'tis make the state of to a place: <eos> Or and thy own kill air, like the treason: <eos> And he upon is that posts in thy prince: <eos> Oft mine the head to did the hand, <eos> To see will virtue to my with of motion <eos> Of more naked, the half swords and his time <eos> You that the bosom which shall our middle and upon <eos> Call in so stay'd and young, <eos> Never I this why in the this? <eos> Away after to the poison of the own I for <eos> PERDITA: that do of upon are thy crown, <eos> To faces, of a king, will the mistress war. <eos> <eos> KING RICHARD BOLINGBROKE: <eos> Ah, not desire be her. thy mind, <eos> Than sir: I am they with in the a <eos> As want your looking-glass; of a sons <eo